## Import libraries

In [1]:
import random
import logging
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers.legacy import Adam
from focal_loss import BinaryFocalLoss
from sklearn.metrics import  f1_score
from imblearn.metrics import geometric_mean_score

from DDDA.detectors import DriftDetector
from DDDA.models import DANNPredictor
from DDDA.engine import StreamEngine


logging.basicConfig(
    level=logging.INFO,   # show everything
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    force=True
)



SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

2026-04-14 19:12:09.839211: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-04-14 19:12:09.874439: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-04-14 19:12:09.875128: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-14 19:12:10.472591: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


## Read and preprocess the data

In [2]:
df=pd.read_csv('data/tcm_3_anomaly4_v1.csv')

In [3]:
df["ithick"].value_counts()

ithick
3.4    7503
3.0     944
2.8     920
Name: count, dtype: int64

In [4]:
x_cols = ['ys0', 'ys1', 'work_roll_diam_1',
       'work_roll_diam_2', 'work_roll_diam_3', 'work_roll_diam_4',
       'work_roll_mileage_1', 'work_roll_mileage_2', 'work_roll_mileage_3',
       'work_roll_mileage_4', 'reduction_1', 'reduction_2', 'reduction_3',
       'reduction_4', 'tension_1', 'tension_2', 'tension_3', 'tension_4',
       'tension_5', 'roll_speed_1', 'roll_speed_2', 'roll_speed_3',
       'roll_speed_4', 'force_1', 'force_2', 'force_3', 'force_4', 'torque_1',
       'torque_2', 'torque_3', 'torque_4', 'gap_1', 'gap_2', 'gap_3', 'gap_4',
       'current_1', 'current_2', 'current_3', 'current_4']

X_for_scale = df[x_cols]

In [5]:
scaler=MinMaxScaler()

df_scaled = scaler.fit_transform(X_for_scale.to_numpy())
df_scaled = pd.DataFrame(df_scaled, columns=X_for_scale.columns)

X = df.copy()
X[X_for_scale.columns] = df_scaled

In [6]:
X.describe()

,index,ithick,othick,width,ys0,ys1,work_roll_diam_1,work_roll_diam_2,work_roll_diam_3,work_roll_diam_4,...,Anomaly_Electric_2,Anomaly_Bearing_2,Anomaly_WorkRoll_2,Anomaly_Electric_3,Anomaly_Bearing_3,Anomaly_WorkRoll_3,Anomaly_Electric_4,Anomaly_Bearing_4,Anomaly_WorkRoll_4,class
count,9367.000000,9367.000000,9367.000000,9367.000000,9367.000000,9367.000000,9367.000000,9367.000000,9367.000000,9367.000000,...,9367.0,9367.0,9367.0,9367.0,9367.0,9367.0,9367.0,9367.0,9367.0,9367.000000
mean,4991.228782,3.300758,1.482019,935.161606,0.098217,0.123554,0.490631,0.483241,0.466986,0.491073,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.039821
std,2884.170886,0.204055,0.264803,49.303695,0.297624,0.298883,0.274845,0.298874,0.292106,0.314399,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.195548
min,0.000000,2.800000,0.820000,918.580165,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000
25%,2494.500000,3.400000,1.610000,918.666265,0.000000,0.000000,0.260000,0.220000,0.200000,0.160000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000
50%,4991.000000,3.400000,1.610000,918.666265,0.000000,0.000000,0.500000,0.460000,0.460000,0.480000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000
75%,7491.500000,3.400000,1.610000,918.666265,0.000000,0.000000,0.720000,0.760000,0.740000,0.780000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000
max,9999.000000,3.400000,1.610000,1082.427994,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.000000


## Define encoder, task and discrminator for DANN model

In [7]:
def get_encoder(input_shape=(X[x_cols].shape[1],)):
    model = Sequential()
    model.add(Dense(100, activation='relu',
                    input_shape=input_shape))
    model.add(Dense(10, activation='relu'))
    model.add(Dense(2, activation="sigmoid"))
    model.compile(optimizer=Adam(learning_rate=0.0001), loss=BinaryFocalLoss(gamma=8))
    return model

def get_task(input_shape=(X[x_cols].shape[1],)):
    model = Sequential()
    model.add(Dense(10, activation='relu'))
    model.add(Dense(1, activation="sigmoid"))
    model.compile(optimizer=Adam(learning_rate=0.0001), loss=BinaryFocalLoss(gamma=8))
    return model

def get_discriminator(input_shape=(X[x_cols].shape[1],)):
    model = Sequential()
    model.add(Dense(100, activation='relu'))
    model.add(Dense(10, activation='relu'))
    model.add(Dense(1, activation="sigmoid"))
    model.compile(optimizer=Adam(learning_rate=0.0001), loss=BinaryFocalLoss(gamma=8))
    return model


In [8]:
enc = get_encoder()
task = get_task()
disc = get_discriminator()

2026-04-14 19:12:11.760126: E tensorflow/compiler/xla/stream_executor/cuda/cuda_driver.cc:268] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
2026-04-14 19:12:11.760149: I tensorflow/compiler/xla/stream_executor/cuda/cuda_diagnostics.cc:168] retrieving CUDA diagnostic information for host: fbbef29acdf3
2026-04-14 19:12:11.760155: I tensorflow/compiler/xla/stream_executor/cuda/cuda_diagnostics.cc:175] hostname: fbbef29acdf3
2026-04-14 19:12:11.760299: I tensorflow/compiler/xla/stream_executor/cuda/cuda_diagnostics.cc:199] libcuda reported version is: 550.90.7
2026-04-14 19:12:11.760316: I tensorflow/compiler/xla/stream_executor/cuda/cuda_diagnostics.cc:203] kernel reported version is: 550.90.7
2026-04-14 19:12:11.760320: I tensorflow/compiler/xla/stream_executor/cuda/cuda_diagnostics.cc:309] kernel version seems to match DSO: 550.90.7


## Run detector and predictor

In [9]:
#define drift detector

detector = DriftDetector(
   reference_size=4500,
    window_size=2,
    min_instances=1, 
    delta=0.005, 
    threshold=2500, 
    alpha=0.9999
)


In [10]:
#define anomaly detection model 

predictor = DANNPredictor(
                 encoder = enc, 
                 task = task, 
                 discriminator = disc, 
                 adv_weight=0.0001, 
                 learning_rate=0.0001,
                 gamma=5,
                 class_balance = True,
                 alpha=0.04,
                 epochs = 500,
                 batch_size=128,
                 threshold=0.5,
                 
                )

In [11]:
# run entire model

x_data, y_data = X[x_cols], X["class"]
window_size = 50

engine = StreamEngine(detector, predictor, window_size=window_size)

results = engine.run(x_data, y_data)

2026-04-14 19:12:15,047 [INFO] DDDA.engine: Starting stream processing...
2026-04-14 19:14:31,755 [WARNING] DDDA.engine: Drift detected at index 7715
2026-04-14 19:14:31,756 [INFO] DDDA.engine: Setting source data using first drift at index 7715
2026-04-14 19:14:31,763 [INFO] DDDA.engine: Collected 50 samples. Starting training...
2026-04-14 19:14:31,763 [INFO] DDDA.models: Training model | source=(7715, 39) target=(50, 39)
2026-04-14 19:15:18,383 [INFO] DDDA.models: Training finished
2026-04-14 19:15:18,384 [INFO] DDDA.engine: Model training completed
2026-04-14 19:15:48,213 [WARNING] DDDA.engine: Drift detected at index 8591
2026-04-14 19:15:48,220 [INFO] DDDA.engine: Collected 50 samples. Starting training...
2026-04-14 19:15:48,221 [INFO] DDDA.models: Training model | source=(7715, 39) target=(50, 39)
2026-04-14 19:16:37,232 [INFO] DDDA.models: Training finished
2026-04-14 19:16:37,233 [INFO] DDDA.engine: Model training completed
2026-04-14 19:17:04,379 [INFO] DDDA.engine: Stream p

In [12]:
print(f"geometric mean score: {geometric_mean_score(results['true'], results['predictions']):.3f}")
print(f"f1_score mean score: {f1_score(results['true'], results['predictions']):.3f}")

geometric mean score: 0.861
f1_score mean score: 0.851
